# M11.2 — Export M9 Step 2 / 09_2B (GAP pooled) → Hub model clone

Plan: [`plans/milestone_11/11_huggingface_artifacts_plan.md`](../../plans/milestone_11/11_huggingface_artifacts_plan.md).  
Architecture: [`plans/00_architecture.md`](../../plans/00_architecture.md) §5.7.

Extracts the **09_2B compact pooled (GAP) head only** (`f2_state`) from
`checkpoints/m9/m09_e2e_pooled_geometry_fusion.pt` into the local clone of
[`tbhugging/camera_orbit_compact_09_2b`](https://huggingface.co/tbhugging/camera_orbit_compact_09_2b).

This is **not** the 09_2A Fourier-pooling variant (`m09_e2e_fourier_geometry_fusion.pt`).

Machine-local staging path: gitignored `configs/hf/local.toml`
(copy from `configs/hf/local.toml.example`). This notebook writes weights + card
into that clone; it does **not** push to the Hub.

In [ ]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,hf,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")

## Resolve local clone + export

Requires:
- `configs/hf/local.toml` with `models.camera_orbit_compact_09_2b.local_clone`
- Study checkpoint `checkpoints/m9/m09_e2e_pooled_geometry_fusion.pt`

In [ ]:
from IPython.display import Markdown, display

from tomography_ml_validation.milestone_11 import (
    export_camera_orbit_compact_09_2b,
    resolve_camera_orbit_compact_09_2b_paths,
)

paths = resolve_camera_orbit_compact_09_2b_paths(ROOT)
display(Markdown(
    f"**Hub:** [`{paths.hub_id}`]({paths.hub_url})  \n"
    f"**Local clone:** `{paths.local_clone}`"
))

result = export_camera_orbit_compact_09_2b(ROOT)
display(Markdown(
    f"Wrote `{display_path(result.weights_path)}`, "
    f"`{display_path(result.config_path)}`, "
    f"`{display_path(result.readme_path)}`  \n"
    f"n_params={result.n_params}  lr={result.lr:g}  "
    f"metrics={{{', '.join(f'{k}={v:.4f}' for k, v in sorted(result.metrics.items()))}}}"
))

## Smoke-check reload

In [ ]:
import json
import torch

from tomography_ml.localization.localize_multiview import (
    GeometryAwareFourierFusionLocalizer,
)

cfg = json.loads(result.config_path.read_text(encoding="utf-8"))
assert cfg["variant_id"] == "m09_2_e2e_pooled_geometry_fusion"
assert cfg["n_views"] == 6

model = GeometryAwareFourierFusionLocalizer.for_09_2_pooled(
    n_views=cfg["n_views"],
    view_angles_deg=cfg["view_angles_deg"],
)
views = torch.zeros(1, cfg["n_views"], 1, cfg["image_height"], cfg["image_width"])
model(views)
state = torch.load(result.weights_path, map_location="cpu", weights_only=True)
model.load_state_dict(state, strict=True)
model.eval()
xyz = model(views)
print("reload ok", tuple(xyz.shape), "config n_params=", cfg["n_params"])

## Hub upload (manual)

After reviewing the local clone:

```bash
hf auth login
cd "$local_clone"   # from configs/hf/local.toml
hf upload tbhugging/camera_orbit_compact_09_2b . .
# or: git add -A && git commit && git push
```